In [4]:
import os
from dotenv import load_dotenv
# from openai import OpenAI
import base64
import json
from PIL import Image
import io
import requests
# import wikipedia
import llama_cpp

In [3]:
load_dotenv()

YC_FOLDER_ID = os.getenv('YC_FOLDER_ID')
YC_API_KEY = os.getenv('YC_API_KEY')

In [11]:
with open('/Users/anastasiya/Documents/AIGuide/setup_data_v2/output/landmarks_filtered_with_name.json', 'r') as f:
    data = json.load(f)

In [ ]:
"images/962_5.jpg",
        "images/5273_1.jpg",
        "images/6166_2.jpg"

In [22]:
for d in data[:5]:
    print(d['name_ru'], d['image_path'])

Театр Делакорта ['14_1.jpg']
Кастильо-де-Санта-Крус ['22_1.jpg', '22_3.jpg', '22_5.jpg']
Церковь Святого Петра, Клейдон ['29_1.jpg', '29_3.jpg', '29_5.jpg']
Церковь Святого Олава ['32_1.jpg', '32_5.jpg']
Раймунд-театр ['50_1.jpg', '50_4.jpg', '50_5.jpg']


In [5]:
data[24]

{'wikidata_id': 'Q4265706',
 'name_en': "St. John's Cathedral",
 'wikidata_description_en': "Anglican cathedral in St. John's, Antigua",
 'coordinates': {'latitude': 17.1227, 'longitude': -61.8419},
 'year_built': '1845',
 'country_ru': 'Антигуа и Барбуда',
 'country_en': 'Antigua and Barbuda',
 'city_ru': 'Сент-Джонс',
 'city_en': "St. John's",
 'website': None,
 'wikipedia_url_en': "https://en.wikipedia.org/wiki/St._John's_Cathedral_(Antigua_and_Barbuda)",
 'landmark_type': {'ru': 'церковь', 'en': 'church', 'type_id': 'церковь'},
 'wikipedia_summary_en': "St. John's Cathedral also known as the Cathedral of St. John the Divine, the Cathedral Church of the Diocese of North Eastern Caribbean and Aruba, is an Anglican church perched on a hilltop in St. John's, Antigua and Barbuda. It is the seat of the Diocese of the North East Caribbean and Aruba in the Church in the Province of the West Indies.",
 'url': "http://commons.wikimedia.org/wiki/Category:St._John's_Cathedral_(Antigua_and_Barb

In [ ]:
image_path = "/Users/anastasiya/Documents/AIGuide/images/475_3.jpg"

with open(image_path, "rb") as image_file:
    encoded_string = base64.b64encode(image_file.read()).decode("utf-8")

# URL и данные для запроса
url = "https://searchapi.api.cloud.yandex.net/v2/image/search_by_image"

payload = {
    "folderId": YC_FOLDER_ID,
    "data": encoded_string
}

headers = {
    "Authorization": f"Api-Key {YC_API_KEY}",
    "Content-Type": "application/json"
}

# Отправка POST-запроса
response = requests.post(url, json=payload, headers=headers)

if response.status_code == 200:
    print("Успешно! Результаты поиска:")
    print(response.json())
else:
    print(f"Ошибка: {response.status_code}")
    print(response.text)

Успешно! Результаты поиска:
{'images': [{'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/St._John%27s-Antigua_%285%29_-_ST._JOHN%27S_CATHEDRAL.jpg/500px-St._John%27s-Antigua_%285%29_-_ST._JOHN%27S_CATHEDRAL.jpg', 'format': 'IMAGE_FORMAT_JPEG', 'width': '736', 'height': '584', 'passage': 'upload.wikimedia.org wikipedia commons 4 47 St._John%27s-Antigua_%285%29_-_ST._JOHN%27S_CATHEDRAL.jpg ', 'host': 'www.pinterest.com', 'pageTitle': 'Belize honeymoon, Belize travel, Honeymoon resorts', 'pageUrl': 'https://www.pinterest.com/pin/belize-honeymoon-belize-travel-honeymoon-resorts--405464772686465132/'}, {'url': 'https://i.pinimg.com/originals/ea/07/2e/ea072ef889099f63973b6083f53d3ab7.jpg', 'format': 'IMAGE_FORMAT_JPEG', 'width': '736', 'height': '460', 'passage': 'Explore the historic St. John&#x27;s Cathedral Church in Antigua, a beautiful old building with two towers and stairs. ', 'host': 'ie.pinterest.com', 'pageTitle': "St. John's Cathedral Church in Antigua", 'pageUrl

In [8]:
res = response.json()

In [10]:
res.keys()

dict_keys(['images', 'maxPage', 'id'])

In [18]:
query = res["images"][1]["pageTitle"]
query

"St. John's Cathedral Church in Antigua"

In [22]:
search_and_describe(query)

"[EN] St. John's Cathedral also known as the Cathedral of St. John the Divine, the Cathedral Church of the Diocese of North Eastern Caribbean and Aruba, is an Anglican church perched on a hilltop in St. John's, Antigua and Barbuda.  It is the seat of the Diocese of the North East Caribbean and Aruba in the Church in the Province of the West Indies.\nThe present cathedral with its imposing white twin towers was built on a fossilized reef, in 1845, and is now in its third incarnation, as earthquakes in 1683 and in 1745 destroyed the previous structures. The iron gates on the south face of the church are flanked by pillars displaying Biblical statues of St John the Divine and St John the Baptist. They were reportedly taken in 1756 from a French ship destined for Martinique."

In [21]:
def search_and_describe(query, prefer_lang='ru', fallback_lang='en'):
    """
    Ищет описание достопримечательности сначала на предпочтительном языке,
    если не удаётся — на запасном.

    Параметры:
    - query (str): Название достопримечательности (на любом языке).
    - prefer_lang (str): Код языка для предпочтительного поиска (по умолч. 'ru').
    - fallback_lang (str): Код запасного языка (по умолч. 'en').

    Возвращает:
    - str: Описание или сообщение об ошибке.
    """
    # Пробуем предпочтительный язык
    description = _try_get_summary(query, prefer_lang)
    if description:
        return f"[{prefer_lang.upper()}] {description}"
    
    # Если не получилось, пробуем запасной язык
    description = _try_get_summary(query, fallback_lang)
    if description:
        return f"[{fallback_lang.upper()}] {description}"
    
    return f"Не удалось найти описание для '{query}' ни на {prefer_lang}, ни на {fallback_lang}."

def _try_get_summary(query, lang):
    """
    Внутренняя функция: пытается получить summary для query на заданном языке.
    Возвращает строку описания или None при любой ошибке.
    """
    try:
        wikipedia.set_lang(lang)
        # Ищем страницу по точному названию (можно заменить на search + первый результат)
        # Используем search для большей гибкости, если точное название не найдено
        search_results = wikipedia.search(query, results=1)
        if not search_results:
            return None
        page = wikipedia.page(search_results[0])
        return page.summary
    except (wikipedia.exceptions.PageError, 
            wikipedia.exceptions.DisambiguationError,
            wikipedia.exceptions.WikipediaException):
        return None
    except Exception:
        return None

# Пример использования
print(search_and_describe("Эйфелева башня"))      # Запрос на русском
print(search_and_describe("Eiffel Tower"))        # Запрос на английском
print(search_and_describe("Колизей"))             # Колизей в Риме
print(search_and_describe("UnknownPlace123"))     # Несуществующее место

[RU] Э́йфелева ба́шня (фр. tour Eiffel, МФА: [tu.ʁ‿ɛ.fɛl]) — металлическая башня в центре Парижа, самая узнаваемая его архитектурная достопримечательность. Названа в честь главного конструктора Гюстава Эйфеля; сам Эйфель называл её просто «300-метровая башня» (tour de 300 mètres).
Башня, впоследствии ставшая символом Парижа, строилась с 1887-го по 1889 год и первоначально задумывалась как временное сооружение, служившее входной аркой парижской Всемирной выставки 1889 года.
Эйфелеву башню называют самой посещаемой платной и самой фотографируемой достопримечательностью мира.


[RU] Э́йфелева ба́шня (фр. tour Eiffel, МФА: [tu.ʁ‿ɛ.fɛl]) — металлическая башня в центре Парижа, самая узнаваемая его архитектурная достопримечательность. Названа в честь главного конструктора Гюстава Эйфеля; сам Эйфель называл её просто «300-метровая башня» (tour de 300 mètres).
Башня, впоследствии ставшая символом Парижа, строилась с 1887-го по 1889 год и первоначально задумывалась как временное сооружение, служ